In [22]:
# Install Flex and GCC
!apt-get update -qq
!apt-get install -y flex gcc -qq

# Create lexer.l
lexer_code = r'''
%{
#include <stdio.h>
%}

KEYWORD int|float|char|double|void|for|while|if|else|return|struct|switch|case|break|do

%%
"#include"                       { printf("Preprocessor Directive : %s\n", yytext); }
"<"[a-zA-Z.]+">"                 { printf("Header File : %s\n", yytext); }
{KEYWORD}                        { printf("Keyword : %s\n", yytext); }
[a-zA-Z_][a-zA-Z0-9_]*           { printf("Identifier : %s\n", yytext); }
[0-9]+                           { printf("Number : %s\n", yytext); }
"=="|"<="|">="                   { printf("Operator : %s\n", yytext); }
"+"|"-"|"*"|"/"|"="|"<"|">"      { printf("Operator : %s\n", yytext); }
[(){};,]                         { printf("Delimiter : %s\n", yytext); }
[ \t\n]+                         { }
.                                { }

%%

int yywrap()
{
    return 1;
}

int main(int argc, char *argv[])
{
    if (argc < 2)
    {
        printf("Usage: %s <input file>\n", argv[0]);
        return 1;
    }

    yyin = fopen(argv[1], "r");

    if (yyin == NULL)
    {
        printf("Error opening input file\n");
        return 1;
    }

    yylex();

    fclose(yyin);

    printf("\nEnd of file\n");

    return 0;
}
'''

# Write lexer.l
with open("lexer.l", "w") as f:
    f.write(lexer_code)

# Create input C program
input_code = r'''
#include <stdio.h>

int main() {
    int a = 10;
    float b = 20;

    if (a < b) {
        a = a + 5;
    }

    return 0;
}
'''

with open("input.c", "w") as f:
    f.write(input_code)

# Generate C code using Flex
!flex lexer.l

# Compile the generated C code
!gcc lex.yy.c -o lexer

# Run the Lexical Analyzer
print("========== LEXICAL ANALYZER OUTPUT ==========\n")
!./lexer input.c

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
========== LEXICAL ANALYZER OUTPUT ==========

Preprocessor Directive : #include
Header File : <stdio.h>
Keyword : int
Identifier : main
Delimiter : (
Delimiter : )
Delimiter : {
Keyword : int
Identifier : a
Operator : =
Number : 10
Delimiter : ;
Keyword : float
Identifier : b
Operator : =
Number : 20
Delimiter : ;
Keyword : if
Delimiter : (
Identifier : a
Operator : <
Identifier : b
Delimiter : )
Delimiter : {
Identifier : a
Operator : =
Identifier : a
Operator : +
Number : 5
Delimiter : ;
Delimiter : }
Keyword : return
Number : 0
Delimiter : ;
Delimiter : }

End of file
